In [2]:
from google.colab import files
uploaded = files.upload()

Saving datasetold.csv to datasetold.csv


In [3]:
import pandas as pd

df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,product_name,weight_num,weight_unit,form,use_type,price_usd
0,Acretin 30 g cream,30,g,cream,Other,17
1,Adol 24 caplets,24,tablets,tablets,Pain killer,23
2,Aggrex 60 tablets,60,tablets,tablets,Other,14
3,Airoplast nan Tape,0,other,tape,Other,19
4,All-Vent 125 ml syrup,125,ml,syrup,Cough syrup / Bronchial,14


In [4]:
!pip install xgboost joblib

In [5]:
# Keep required columns
df = df[["product_name", "weight_num", "weight_unit", "form", "use_type", "price_usd"]]

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Convert weight_num to numeric
df["weight_num"] = pd.to_numeric(df["weight_num"], errors="coerce")
df = df.dropna(subset=["weight_num", "price_usd"])

# Fill missing categorical values
for col in ["product_name", "weight_unit", "form", "use_type"]:
    df[col] = df[col].fillna("Unknown").astype(str)

df.head()


,product_name,weight_num,weight_unit,form,use_type,price_usd
0,Acretin 30 g cream,30,g,cream,Other,17
1,Adol 24 caplets,24,tablets,tablets,Pain killer,23
2,Aggrex 60 tablets,60,tablets,tablets,Other,14
3,Airoplast nan Tape,0,other,tape,Other,19
4,All-Vent 125 ml syrup,125,ml,syrup,Cough syrup / Bronchial,14


In [6]:
FEATURE_COLS = ["product_name", "weight_num", "weight_unit", "form", "use_type"]
TARGET_COL = "price_usd"

X = df[FEATURE_COLS]
y = df[TARGET_COL]


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

numeric_features = ["weight_num"]
categorical_features = ["product_name", "weight_unit", "form", "use_type"]

preprocessor = ColumnTransformer(
    [
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", xgb_model)
])



In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['weight_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_name',
                                                   'weight_unit', 'form',
                                                   'use_type'])])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.8, device=None,
                              ea...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)


RMSE: 8.329639175495258
MAE : 7.111500263214111
R²  : -0.7532726526260376


In [11]:
model.fit(X, y)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['weight_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_name',
                                                   'weight_unit', 'form',
                                                   'use_type'])])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.8, device=None,
                              ea...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)

RMSE: 2.0269372786165123
MAE : 1.8758894205093384
R²  : 0.8961808085441589


In [14]:
import joblib

joblib.dump(model, "xgboost_price_model.joblib")

['xgboost_price_model.joblib']

In [15]:
from google.colab import files
files.download("xgboost_price_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import pandas as pd
import joblib

loaded = joblib.load("xgboost_price_model.joblib")

example = pd.DataFrame({
    "product_name": ["C Zinc"],
    "weight_num": [30],
    "weight_unit": ["capsules"],
    "form": ["capsules"],
    "use_type": ["immunity"]
})

price = loaded.predict(example)[0]
print("Predicted price:", float(price))


Predicted price: 26.321533203125
